<a href="https://colab.research.google.com/github/loan1/btxrd-stage1-semisup/blob/main/SAM_MEDSAM_BTXRD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/BTXRD"
RAW_DIR = f"{DRIVE_ROOT}/raw"
PROC_DIR = f"{DRIVE_ROOT}/processed"
PSEUDO_DIR = f"{DRIVE_ROOT}/pseudo"
RUNS_DIR = f"{DRIVE_ROOT}/runs"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip -q install pandas openpyxl opencv-python pillow tqdm scikit-learn


In [ ]:
%%writefile /content/data_prepare.py
import os, json
import pandas as pd
import numpy as np
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split

def _cat(row):
    if row["tumor"] == 0:
        return "normal"
    if row.get("malignant", 0) == 1:
        return "malignant"
    return "benign"

def load_meta(xlsx_path):
    df = pd.read_excel(xlsx_path)
    df["cat"] = df.apply(_cat, axis=1)
    df["strat"] = df["center"].astype(str) + "_" + df["cat"]
    return df

def split_train_val_test(df, seed=42, test_size=0.2, val_size=0.1):
    train_val, test = train_test_split(
        df, test_size=test_size, random_state=seed, stratify=df["strat"]
    )
    # val_size là tỉ lệ trên toàn bộ => tỉ lệ trên train_val:
    val_ratio_in_trainval = val_size / (1 - test_size)
    train, val = train_test_split(
        train_val, test_size=val_ratio_in_trainval, random_state=seed, stratify=train_val["strat"]
    )
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

def split_labeled_unlabeled(train_df, labeled_frac=0.1, seed=42):
    # labeled chỉ lấy trong tumor
    tumor_df = train_df[train_df["tumor"] == 1].copy()
    normal_df = train_df[train_df["tumor"] == 0].copy()

    tumor_df["strat2"] = tumor_df["center"].astype(str) + "_" + tumor_df["cat"]
    labeled, unlabeled_tumor = train_test_split(
        tumor_df,
        test_size=(1 - labeled_frac),
        random_state=seed,
        stratify=tumor_df["strat2"]
    )
    unlabeled = pd.concat([unlabeled_tumor, normal_df], axis=0).sample(frac=1, random_state=seed)
    return labeled.reset_index(drop=True), unlabeled.reset_index(drop=True)

def labelme_to_mask_and_box(json_path, img_h, img_w):
    with open(json_path, "r") as f:
        ann = json.load(f)

    H0, W0 = ann.get("imageHeight", img_h), ann.get("imageWidth", img_w)
    sx = img_w / float(W0)
    sy = img_h / float(H0)

    mask = np.zeros((img_h, img_w), dtype=np.uint8)

    # lấy polygon(s) => mask; nếu không có polygon thì fallback bbox từ rectangle
    polys = []
    rects = []
    for sh in ann.get("shapes", []):
        pts = np.array(sh["points"], dtype=np.float32)
        pts[:, 0] *= sx
        pts[:, 1] *= sy
        if sh.get("shape_type") == "polygon":
            polys.append(pts.astype(np.int32))
        elif sh.get("shape_type") == "rectangle":
            rects.append(pts)

    if polys:
        cv2.fillPoly(mask, polys, 1)
        ys, xs = np.where(mask > 0)
        x1, y1, x2, y2 = int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())
        box = [x1, y1, x2, y2]
        return mask, box

    # rectangle fallback
    if rects:
        p = rects[0]
        x1, y1 = p[0]
        x2, y2 = p[1]
        x1, y1, x2, y2 = map(int, [min(x1,x2), min(y1,y2), max(x1,x2), max(y1,y2)])
        box = [x1, y1, x2, y2]
        mask[:] = 0
        return mask, box

    return mask, None

def prepare_btxrd(
    raw_images_dir, raw_ann_dir, xlsx_path,
    out_masks_dir, out_boxes_dir, out_splits_dir,
    labeled_frac=0.1, seed=42
):
    os.makedirs(out_masks_dir, exist_ok=True)
    os.makedirs(out_boxes_dir, exist_ok=True)
    os.makedirs(out_splits_dir, exist_ok=True)

    df = load_meta(xlsx_path)
    train, val, test = split_train_val_test(df, seed=seed)
    labeled, unlabeled = split_labeled_unlabeled(train, labeled_frac=labeled_frac, seed=seed)

    # save splits
    def _save_list(name, subdf):
        p = os.path.join(out_splits_dir, name)
        subdf["image_id"].to_csv(p, index=False, header=False)

    _save_list("train.txt", train)
    _save_list("val.txt", val)
    _save_list("test.txt", test)
    _save_list("train_labeled.txt", labeled)
    _save_list("train_unlabeled.txt", unlabeled)

    # build GT masks + boxes (cho tumor); normal => mask rỗng (all zeros), box None
    boxes_records = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Building masks/boxes"):
        img_id = row["image_id"]
        img_path = os.path.join(raw_images_dir, img_id)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(img_path)
        h, w = img.shape

        if row["tumor"] == 1:
            json_path = os.path.join(raw_ann_dir, img_id.replace(".jpeg", ".json").replace(".jpg", ".json"))
            if not os.path.exists(json_path):
                # một số dataset có thể dùng .jpeg cố định, bạn có thể chỉnh thêm nếu cần
                raise FileNotFoundError(json_path)
            mask, box = labelme_to_mask_and_box(json_path, h, w)
        else:
            mask = np.zeros((h, w), dtype=np.uint8)
            box = None

        # lưu mask dạng PNG (0/255)
        out_mask_path = os.path.join(out_masks_dir, img_id.replace(".jpeg", ".png").replace(".jpg", ".png"))
        cv2.imwrite(out_mask_path, (mask * 255).astype(np.uint8))

        boxes_records.append({"image_id": img_id, "box": box})

    pd.DataFrame(boxes_records).to_json(os.path.join(out_boxes_dir, "boxes.json"), orient="records", indent=2)
    print("Done. Splits & GT masks created.")

Writing /content/data_prepare.py


In [ ]:
!pip -q install git+https://github.com/facebookresearch/segment-anything.git
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O /content/sam_vit_b.pth
!pip -q install opencv-python matplotlib

  Preparing metadata (setup.py) ... done


In [ ]:
%%writefile /content/pseudo_sam.py
import os, json
import numpy as np
import cv2
from tqdm import tqdm
import torch
from segment_anything import sam_model_registry, SamPredictor

def load_boxes(boxes_json_path):
    with open(boxes_json_path, "r") as f:
        recs = json.load(f)
    return {r["image_id"]: r["box"] for r in recs}

@torch.no_grad()
def gen_pseudo_sam(images_dir, list_txt, boxes_json, out_dir, ckpt_path, model_type="vit_b"):
    os.makedirs(out_dir, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    sam = sam_model_registry[model_type](checkpoint=ckpt_path)
    sam.to(device=device)
    predictor = SamPredictor(sam)

    boxes = load_boxes(boxes_json)

    with open(list_txt, "r") as f:
        ids = [line.strip() for line in f if line.strip()]

    for img_id in tqdm(ids, desc="Pseudo SAM"):
        box = boxes.get(img_id, None)
        if box is None:
            # normal image => skip hoặc lưu mask rỗng
            continue

        img_path = os.path.join(images_dir, img_id)
        img_gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img_gray is None:
            continue
        img_rgb = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB)

        predictor.set_image(img_rgb)

        box_arr = np.array(box, dtype=np.float32)
        masks, scores, _ = predictor.predict(
            box=box_arr[None, :],
            multimask_output=True
        )
        best = masks[np.argmax(scores)].astype(np.uint8) * 255

        out_path = os.path.join(out_dir, img_id.replace(".jpeg", ".png").replace(".jpg", ".png"))
        cv2.imwrite(out_path, best)

    print("Done pseudo SAM:", out_dir)

Writing /content/pseudo_sam.py


In [ ]:
import os

print("RAW_DIR:", RAW_DIR)
print("PROC_DIR:", PROC_DIR)
print("PSEUDO_DIR:", PSEUDO_DIR)

print("Drive mounted?", os.path.exists("/content/drive/MyDrive"))
print("PROC_DIR exists?", os.path.exists(PROC_DIR))
print("boxes folder exists?", os.path.exists(f"{PROC_DIR}/boxes"))
print("boxes.json exists?", os.path.exists(f"{PROC_DIR}/boxes/boxes.json"))

RAW_DIR: /content/drive/MyDrive/BTXRD/raw
PROC_DIR: /content/drive/MyDrive/BTXRD/processed
PSEUDO_DIR: /content/drive/MyDrive/BTXRD/pseudo
Drive mounted? True
PROC_DIR exists? True
boxes folder exists? True
boxes.json exists? True


In [ ]:
!ls -lah /content/drive/MyDrive/BTXRD/processed
!ls -lah /content/drive/MyDrive/BTXRD/processed/splits

total 12K
drwx------ 2 root root 4.0K Feb 28 14:19 boxes
drwx------ 2 root root 4.0K Feb 28 14:18 masks_gt
drwx------ 2 root root 4.0K Feb 28 14:19 splits
total 94K
-rw------- 1 root root  11K Mar  1 22:04 test.txt
-rw------- 1 root root 2.0K Mar  1 22:04 train_labeled.txt
-rw------- 1 root root  39K Mar  1 22:04 train.txt
-rw------- 1 root root  37K Mar  1 22:04 train_unlabeled.txt
-rw------- 1 root root 5.5K Mar  1 22:04 val.txt


In [ ]:
!ls "/content/drive/MyDrive/BTXRD/raw/images" | wc -l

3746


In [ ]:
%%writefile /content/data_prepare.py
import os, json, glob
import pandas as pd
import numpy as np
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split

def _cat(row):
    if row["tumor"] == 0:
        return "normal"
    if row.get("malignant", 0) == 1:
        return "malignant"
    return "benign"

def load_meta(xlsx_path):
    df = pd.read_excel(xlsx_path)
    # đảm bảo có cột image_id
    if "image_id" not in df.columns:
        raise ValueError("dataset.xlsx phải có cột 'image_id'")
    df["cat"] = df.apply(_cat, axis=1)
    df["strat"] = df["center"].astype(str) + "_" + df["cat"]
    return df

def split_train_val_test(df, seed=42, test_size=0.2, val_size=0.1):
    train_val, test = train_test_split(
        df, test_size=test_size, random_state=seed, stratify=df["strat"]
    )
    val_ratio_in_trainval = val_size / (1 - test_size)
    train, val = train_test_split(
        train_val, test_size=val_ratio_in_trainval, random_state=seed, stratify=train_val["strat"]
    )
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

def split_labeled_unlabeled(train_df, labeled_frac=0.1, seed=42):
    tumor_df = train_df[train_df["tumor"] == 1].copy()
    normal_df = train_df[train_df["tumor"] == 0].copy()

    tumor_df["strat2"] = tumor_df["center"].astype(str) + "_" + tumor_df["cat"]
    labeled, unlabeled_tumor = train_test_split(
        tumor_df,
        test_size=(1 - labeled_frac),
        random_state=seed,
        stratify=tumor_df["strat2"]
    )
    unlabeled = pd.concat([unlabeled_tumor, normal_df], axis=0).sample(frac=1, random_state=seed)
    return labeled.reset_index(drop=True), unlabeled.reset_index(drop=True)

def build_stem_map(images_dir):
    exts = ["jpg","jpeg","JPG","JPEG","png","PNG"]  # phòng hờ
    files = []
    for e in exts:
        files += glob.glob(os.path.join(images_dir, f"*.{e}"))
    stem_map = {}
    for p in files:
        stem = os.path.splitext(os.path.basename(p))[0]  # IMG000987
        stem_map[stem] = p
    return stem_map

def labelme_to_mask_and_box(json_path, img_h, img_w):
    with open(json_path, "r") as f:
        ann = json.load(f)

    H0, W0 = ann.get("imageHeight", img_h), ann.get("imageWidth", img_w)
    sx = img_w / float(W0)
    sy = img_h / float(H0)

    mask = np.zeros((img_h, img_w), dtype=np.uint8)

    polys = []
    rects = []
    for sh in ann.get("shapes", []):
        pts = np.array(sh["points"], dtype=np.float32)
        pts[:, 0] *= sx
        pts[:, 1] *= sy
        if sh.get("shape_type") == "polygon":
            polys.append(pts.astype(np.int32))
        elif sh.get("shape_type") == "rectangle":
            rects.append(pts)

    if polys:
        cv2.fillPoly(mask, polys, 1)
        ys, xs = np.where(mask > 0)
        x1, y1, x2, y2 = int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())
        return mask, [x1, y1, x2, y2]

    if rects:
        p = rects[0]
        x1, y1 = p[0]
        x2, y2 = p[1]
        x1, y1, x2, y2 = map(int, [min(x1,x2), min(y1,y2), max(x1,x2), max(y1,y2)])
        return mask, [x1, y1, x2, y2]

    return mask, None

def prepare_btxrd(
    raw_images_dir, raw_ann_dir, xlsx_path,
    out_masks_dir, out_boxes_dir, out_splits_dir,
    labeled_frac=0.1, seed=42
):
    os.makedirs(out_masks_dir, exist_ok=True)
    os.makedirs(out_boxes_dir, exist_ok=True)
    os.makedirs(out_splits_dir, exist_ok=True)

    df = load_meta(xlsx_path)
    train, val, test = split_train_val_test(df, seed=seed)
    labeled, unlabeled = split_labeled_unlabeled(train, labeled_frac=labeled_frac, seed=seed)

    def _save_list(name, subdf):
        p = os.path.join(out_splits_dir, name)
        subdf["image_id"].to_csv(p, index=False, header=False)

    _save_list("train.txt", train)
    _save_list("val.txt", val)
    _save_list("test.txt", test)
    _save_list("train_labeled.txt", labeled)
    _save_list("train_unlabeled.txt", unlabeled)

    # map ảnh theo stem để tránh lỗi .jpg/.jpeg
    stem_map = build_stem_map(raw_images_dir)

    boxes_records = []
    missing_imgs = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Building masks/boxes"):
        img_id = str(row["image_id"])
        stem = os.path.splitext(img_id)[0]          # IMG000987
        img_path = stem_map.get(stem, None)

        if img_path is None:
            missing_imgs.append(img_id)
            continue

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            missing_imgs.append(img_id)
            continue

        h, w = img.shape

        if row["tumor"] == 1:
            json_path = os.path.join(raw_ann_dir, f"{stem}.json")
            if not os.path.exists(json_path):
                raise FileNotFoundError(f"Missing annotation: {json_path}")
            mask, box = labelme_to_mask_and_box(json_path, h, w)
        else:
            mask = np.zeros((h, w), dtype=np.uint8)
            box = None

        out_mask_path = os.path.join(out_masks_dir, f"{stem}.png")
        cv2.imwrite(out_mask_path, (mask * 255).astype(np.uint8))

        boxes_records.append({"image_id": os.path.basename(img_path), "box": box})

    pd.DataFrame(boxes_records).to_json(os.path.join(out_boxes_dir, "boxes.json"), orient="records", indent=2)

    if missing_imgs:
        miss_path = os.path.join(out_splits_dir, "missing_images.txt")
        with open(miss_path, "w") as f:
            for m in missing_imgs:
                f.write(m + "\n")
        print(f"[WARN] Missing {len(missing_imgs)} images. List saved to: {miss_path}")

    print("Done. Splits & GT masks created.")

Overwriting /content/data_prepare.py


In [ ]:
from data_prepare import prepare_btxrd

prepare_btxrd(
    raw_images_dir=f"{RAW_DIR}/images",
    raw_ann_dir=f"{RAW_DIR}/Annotations",
    xlsx_path=f"{RAW_DIR}/dataset.xlsx",
    out_masks_dir=f"{PROC_DIR}/masks_gt",
    out_boxes_dir=f"{PROC_DIR}/boxes",
    out_splits_dir=f"{PROC_DIR}/splits",
    labeled_frac=0.10,
    seed=42
)

Building masks/boxes:  34%|███▍      | 1292/3746 [30:41<1:01:57,  1.51s/it]

In [ ]:
import pandas as pd, os, random, cv2, numpy as np

df = pd.read_excel(f"{RAW_DIR}/dataset.xlsx")
tumor_df = df[df["tumor"]==1].sample(5, random_state=0)
tumor_df[["image_id","tumor","benign","malignant","center"]]

In [ ]:
import os, cv2, numpy as np

for img_id in tumor_df["image_id"].tolist():
    stem = os.path.splitext(img_id)[0]
    mask_path = f"{PROC_DIR}/masks_gt/{stem}.png"
    m = cv2.imread(mask_path, 0)
    print(img_id, "mask_exists=", os.path.exists(mask_path),
          "shape=", None if m is None else m.shape,
          "max=", None if m is None else int(m.max()),
          "sum=", None if m is None else int(m.sum()))

In [ ]:
import matplotlib.pyplot as plt
import cv2, os, numpy as np

img_id = tumor_df["image_id"].iloc[0]
stem = os.path.splitext(img_id)[0]

# tìm đúng file ảnh thật (.jpg/.jpeg)
import glob
img_candidates = glob.glob(f"{RAW_DIR}/images/{stem}.*")
img_path = img_candidates[0]

img = cv2.imread(img_path, 0)
mask = cv2.imread(f"{PROC_DIR}/masks_gt/{stem}.png", 0)

plt.figure(figsize=(6,6))
plt.imshow(img, cmap="gray")
plt.imshow(mask, alpha=0.35)   # vùng trắng sẽ nổi lên
plt.title(img_id)
plt.axis("off")
plt.show()

In [ ]:
import glob, cv2, numpy as np

paths = glob.glob(f"{PROC_DIR}/masks_gt/*.png")
non_empty = 0
for p in paths:
    m = cv2.imread(p, 0)
    if m is not None and m.max() > 0:
        non_empty += 1

print("Total masks:", len(paths))
print("Non-empty masks:", non_empty)

In [ ]:
from pseudo_sam import gen_pseudo_sam

gen_pseudo_sam(
    images_dir=f"{RAW_DIR}/images",
    list_txt=f"{PROC_DIR}/splits/train_unlabeled.txt",
    boxes_json=f"{PROC_DIR}/boxes/boxes.json",
    out_dir=f"{PSEUDO_DIR}/sam_box_oracle",
    ckpt_path="/content/sam_vit_b.pth",
    model_type="vit_b"
)

In [ ]:
import os, glob

print("Labeled list:", sum(1 for _ in open(f"{PROC_DIR}/splits/train_labeled.txt")))
print("Unlabeled list:", sum(1 for _ in open(f"{PROC_DIR}/splits/train_unlabeled.txt")))
print("Val list:", sum(1 for _ in open(f"{PROC_DIR}/splits/val.txt")))
print("Test list:", sum(1 for _ in open(f"{PROC_DIR}/splits/test.txt")))

print("GT masks:", len(glob.glob(f"{PROC_DIR}/masks_gt/*.png")))
print("SAM pseudo masks:", len(glob.glob(f'{PSEUDO_DIR}/sam_box_oracle/*.png')))

In [ ]:
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install opencv-python pillow tqdm matplotlib

In [ ]:
%%writefile btxrd_dataset.py
import os, glob
import cv2
import numpy as np
from torch.utils.data import Dataset

def find_image_path(images_dir, image_id):
    stem = os.path.splitext(image_id)[0]
    cand = glob.glob(os.path.join(images_dir, stem + ".*"))
    if not cand:
        raise FileNotFoundError(f"Missing image for {image_id}")
    return cand[0], stem

class BTXRDSegDataset(Dataset):
    def __init__(self, images_dir, ids_list, mask_mode,
                 gt_masks_dir, pseudo_masks_dir=None, img_size=512):
        """
        mask_mode:
          - 'gt'    : dùng GT mask (train_labeled, val, test)
          - 'pseudo': dùng pseudo nếu có, không có => mask rỗng (train_unlabeled)
        """
        self.images_dir = images_dir
        self.ids_list = ids_list
        self.mask_mode = mask_mode
        self.gt_masks_dir = gt_masks_dir
        self.pseudo_masks_dir = pseudo_masks_dir
        self.img_size = img_size

    def __len__(self):
        return len(self.ids_list)

    def _load_mask(self, stem, img_shape):
        if self.mask_mode == "gt":
            mpath = os.path.join(self.gt_masks_dir, stem + ".png")
            m = cv2.imread(mpath, 0)
            if m is None:
                raise FileNotFoundError(mpath)
            return m

        # pseudo mode
        ppath = os.path.join(self.pseudo_masks_dir, stem + ".png")
        m = cv2.imread(ppath, 0)
        if m is None:
            return np.zeros(img_shape, dtype=np.uint8)  # normal => empty
        return m

    def __getitem__(self, idx):
        image_id = self.ids_list[idx]
        img_path, stem = find_image_path(self.images_dir, image_id)

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(img_path)

        mask = self._load_mask(stem, img.shape)

        # resize về cùng size để train
        img_rs = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
        mask_rs = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)

        img_rs = img_rs.astype(np.float32) / 255.0
        mask_rs = (mask_rs > 127).astype(np.float32)

        img_rs = img_rs[None, :, :]     # (1,H,W)
        mask_rs = mask_rs[None, :, :]   # (1,H,W)
        return img_rs, mask_rs, image_id

In [ ]:
%%writefile unet.py
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=32):
        super().__init__()
        self.d1 = DoubleConv(in_ch, base)
        self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(base, base*2)
        self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(base*2, base*4)
        self.p3 = nn.MaxPool2d(2)
        self.d4 = DoubleConv(base*4, base*8)

        self.u3 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2)
        self.c3 = DoubleConv(base*8, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2)
        self.c2 = DoubleConv(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.c1 = DoubleConv(base*2, base)

        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        x1 = self.d1(x)
        x2 = self.d2(self.p1(x1))
        x3 = self.d3(self.p2(x2))
        x4 = self.d4(self.p3(x3))

        y = self.u3(x4)
        y = self.c3(torch.cat([y, x3], dim=1))
        y = self.u2(y)
        y = self.c2(torch.cat([y, x2], dim=1))
        y = self.u1(y)
        y = self.c1(torch.cat([y, x1], dim=1))
        return self.out(y)  # logits

In [ ]:
%%writefile train_unet.py
import os, time
import torch
import torch.nn as nn
from tqdm import tqdm

def dice_from_logits(logits, target, eps=1e-6):
    prob = torch.sigmoid(logits)
    pred = (prob > 0.5).float()
    inter = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    dice = (2*inter + eps) / (union + eps)
    return dice.mean()

def iou_from_logits(logits, target, eps=1e-6):
    prob = torch.sigmoid(logits)
    pred = (prob > 0.5).float()
    inter = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) - inter
    iou = (inter + eps) / (union + eps)
    return iou.mean()

def dice_loss_with_logits(logits, target, eps=1e-6):
    prob = torch.sigmoid(logits)
    prob = prob.view(prob.size(0), -1)
    target = target.view(target.size(0), -1)
    inter = (prob * target).sum(dim=1)
    union = prob.sum(dim=1) + target.sum(dim=1)
    dice = (2*inter + eps) / (union + eps)
    return 1 - dice.mean()

def train_one_epoch(model, loader, opt, device):
    model.train()
    bce = nn.BCEWithLogitsLoss()
    total = 0.0
    for x, y, _ in tqdm(loader, desc="Train", leave=False):
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        logits = model(x)
        loss = 0.5*bce(logits, y) + 0.5*dice_loss_with_logits(logits, y)
        loss.backward()
        opt.step()
        total += loss.item() * x.size(0)
    return total / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    dices, ious = [], []
    for x, y, _ in tqdm(loader, desc="Eval", leave=False):
        x, y = x.to(device), y.to(device)
        logits = model(x)
        dices.append(dice_from_logits(logits, y).item())
        ious.append(iou_from_logits(logits, y).item())
    return sum(dices)/len(dices), sum(ious)/len(ious)

def fit(model, train_loader, val_loader, device, out_dir, lr=1e-3, epochs=20):
    os.makedirs(out_dir, exist_ok=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    best = -1
    for ep in range(1, epochs+1):
        t0 = time.time()
        tr = train_one_epoch(model, train_loader, opt, device)
        vd, vi = evaluate(model, val_loader, device)
        print(f"Epoch {ep:02d} | loss={tr:.4f} | val_dice={vd:.4f} | val_iou={vi:.4f} | {time.time()-t0:.1f}s")

        if vd > best:
            best = vd
            torch.save(model.state_dict(), os.path.join(out_dir, "best.pth"))

    print("Best val dice:", best)

In [ ]:
import torch
from torch.utils.data import DataLoader
from btxrd_dataset import BTXRDSegDataset
from unet import UNet
from train_unet import fit, evaluate

def read_list(p):
    return [x.strip() for x in open(p) if x.strip()]

train_labeled = read_list(f"{PROC_DIR}/splits/train_labeled.txt")
val_ids = read_list(f"{PROC_DIR}/splits/val.txt")
test_ids = read_list(f"{PROC_DIR}/splits/test.txt")

device = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 512
BATCH = 4
NW = 0  # Colab đôi khi lỗi worker -> để 0 cho chắc

train_sup = BTXRDSegDataset(
    images_dir=f"{RAW_DIR}/images",
    ids_list=train_labeled,
    mask_mode="gt",
    gt_masks_dir=f"{PROC_DIR}/masks_gt",
    img_size=IMG_SIZE
)
val_set = BTXRDSegDataset(
    images_dir=f"{RAW_DIR}/images",
    ids_list=val_ids,
    mask_mode="gt",
    gt_masks_dir=f"{PROC_DIR}/masks_gt",
    img_size=IMG_SIZE
)

train_loader = DataLoader(train_sup, batch_size=BATCH, shuffle=True, num_workers=NW)
val_loader = DataLoader(val_set, batch_size=BATCH, shuffle=False, num_workers=NW)

model = UNet(in_ch=1, out_ch=1, base=32).to(device)
outA = f"{RUNS_DIR}/unet_supervised_p10"
fit(model, train_loader, val_loader, device, outA, lr=1e-3, epochs=20)

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os, glob
RUNS_DIR = "/content/drive/MyDrive/BTXRD/runs"

out_dir = f"{RUNS_DIR}/unet_supervised_p10_moreNorm_posw15"
print("Files:", sorted([os.path.basename(p) for p in glob.glob(out_dir+"/*")]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files: ['best.pth']


In [3]:
import os
import torch
import torch.nn as nn
from tqdm import tqdm
from contextlib import nullcontext

def dice_loss_with_logits(logits, target, eps=1e-6):
    prob = torch.sigmoid(logits).view(logits.size(0), -1)
    target = target.view(target.size(0), -1)
    inter = (prob * target).sum(dim=1)
    union = prob.sum(dim=1) + target.sum(dim=1)
    dice = (2*inter + eps) / (union + eps)
    return 1 - dice.mean()

@torch.no_grad()
def eval_tumor_only(model, loader, device, thr=0.5, eps=1e-6):
    model.eval()
    dice_t = []
    for x, y, _ in loader:
        x, y = x.to(device), y.to(device)
        prob = torch.sigmoid(model(x))
        pred = (prob > thr).float()
        inter = (pred * y).sum(dim=(1,2,3))
        sum_pred = pred.sum(dim=(1,2,3))
        sum_gt = y.sum(dim=(1,2,3))
        dice = (2*inter + eps) / (sum_pred + sum_gt + eps)
        has_tumor = (sum_gt > 0)
        if has_tumor.any():
            dice_t.append(dice[has_tumor].mean().item())
    return sum(dice_t)/len(dice_t) if len(dice_t)>0 else 0.0

def fit_posw_resume(
    model, train_loader, val_loader, device, out_dir,
    pos_weight, lr=1e-3, epochs_total=20, thr_val=0.5,
    resume=True
):
    os.makedirs(out_dir, exist_ok=True)

    ckpt_last = os.path.join(out_dir, "last.pt")
    ckpt_best = os.path.join(out_dir, "best.pth")

    opt = torch.optim.Adam(model.parameters(), lr=lr)

    bce = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_weight], device=device)
    )

    use_amp = device.startswith("cuda")
    if use_amp:
        from torch.amp import autocast, GradScaler
        scaler = GradScaler("cuda")
        amp_ctx = lambda: autocast(device_type="cuda", dtype=torch.float16)
    else:
        scaler = None
        amp_ctx = nullcontext

    start_epoch = 1
    best_val = -1.0

    # ---- resume logic
    if resume and os.path.exists(ckpt_last):
        ck = torch.load(ckpt_last, map_location=device)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        if use_amp and ck.get("scaler") is not None:
            scaler.load_state_dict(ck["scaler"])
        start_epoch = ck["epoch"] + 1
        best_val = ck["best_val"]
        print(f"[RESUME] from last.pt at epoch {ck['epoch']} -> continue from {start_epoch}")
    elif resume and os.path.exists(ckpt_best):
        # fallback: resume from best weights only (optimizer resets)
        model.load_state_dict(torch.load(ckpt_best, map_location=device))
        print("[RESUME] last.pt not found, loaded best.pth (optimizer reset)")

    for ep in range(start_epoch, epochs_total + 1):
        model.train()
        total = 0.0

        for x, y, _ in tqdm(train_loader, desc=f"Train {ep}", leave=False):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)

            with amp_ctx():
                logits = model(x)
                loss = 0.5*bce(logits, y) + 0.5*dice_loss_with_logits(logits, y)

            if use_amp:
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                opt.step()

            total += loss.item() * x.size(0)

        val_dice_t = eval_tumor_only(model, val_loader, device, thr=thr_val)
        print(f"Epoch {ep:02d} | loss={total/len(train_loader.dataset):.4f} | val_dice_tumor@{thr_val}={val_dice_t:.4f}")

        # save last every epoch
        torch.save({
            "epoch": ep,
            "model": model.state_dict(),
            "opt": opt.state_dict(),
            "scaler": scaler.state_dict() if use_amp else None,
            "best_val": best_val,
        }, ckpt_last)

        # save best
        if val_dice_t > best_val:
            best_val = val_dice_t
            torch.save(model.state_dict(), ckpt_best)

    print("Done. Best val_dice_tumor:", best_val, "best saved:", ckpt_best, "last saved:", ckpt_last)

In [4]:
# (recreate train_loader_more_norm, val_loader như bạn đã làm trước đó)

model = UNet(1,1,32).to(device)

out_dir = f"{RUNS_DIR}/unet_supervised_p10_moreNorm_posw15"
fit_posw_resume(
    model,
    train_loader_more_norm,
    val_loader,
    device,
    out_dir,
    pos_weight=15.0,
    lr=1e-3,
    epochs_total=20,   # tổng epoch bạn muốn đạt tới
    thr_val=0.5,
    resume=True
)

NameError: name 'UNet' is not defined

In [6]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/BTXRD"
RAW_DIR  = f"{DRIVE_ROOT}/raw"
PROC_DIR = f"{DRIVE_ROOT}/processed"
RUNS_DIR = f"{DRIVE_ROOT}/runs"

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device: cuda


In [8]:
!find /content/drive/MyDrive -maxdepth 5 -name "btxrd_dataset.py" -print
!find /content/drive/MyDrive -maxdepth 5 -name "unet.py" -print